# Dynamic Fleet Failure & Delay Forecasting System

Production-style notebook using modules in `src/`.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(ROOT))

from src.config import PROJECT
from src.simulate import simulate_system
from src.features import add_rolling_features, make_composite_observation
from src.kalman import kalman_filter_1d
from src.hybrid_model import train_hybrid_model, score_dataframe
from src.causal_analysis import estimate_intervention_effect


In [ ]:
df = simulate_system(PROJECT['observable_signals'])
df = make_composite_observation(df)
x_kf, P = kalman_filter_1d(df['composite_obs'].values)
df['x_kf'] = x_kf
df['x_kf_std'] = P ** 0.5
df = add_rolling_features(df, PROJECT['observable_signals'][:3])
df.head()


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df['t'], df['x_true'], label='x_true')
plt.plot(df['t'], df['x_kf'], label='x_kf')
plt.fill_between(df['t'], df['x_kf'] - 2*df['x_kf_std'], df['x_kf'] + 2*df['x_kf_std'], alpha=0.2)
plt.legend()
plt.title(PROJECT['title'] + ' — latent state estimation')
plt.show()


In [ ]:
model, feature_set, auc, report = train_hybrid_model(df)
df = score_dataframe(model, df, feature_set)
print('AUC:', auc)
print(report)


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df['t'], df['p_event_ml'], label='Predicted event probability')
plt.legend()
plt.title(PROJECT['title'] + ' — hybrid risk score')
plt.show()


In [ ]:
res, ate = estimate_intervention_effect(df)
print('Estimated intervention effect:', ate)
print(res.summary())
